# Chapitre 7 - Cours 1 : ACID, atomicité et durabilité

Ce notebook couvre la première grande section du chapitre, *The Slippery Concept of a
Transaction* : l'introduction du chapitre, *The Meaning of ACID* (*Atomicity*,
*Consistency*, *Isolation*, *Durability*, l'encadré *Replication and Durability*) et
*Single-Object and Multi-Object Operations* (*Single-object writes*, *The need for
multi-object transactions*, *Handling errors and aborts*). Les niveaux d'isolation sont
l'objet des cours 2 à 4 ; la validation atomique distribuée (2PC) sera traitée au
chapitre 9.

**Plan.**

1. Pourquoi des transactions ? Ce qui peut mal tourner.
2. ACID lettre par lettre (et pourquoi le « C » est à part).
3. Atomicité : un journal pour faire « tout ou rien ».
4. Durabilité : fsync, validation groupée et réplication.
5. Opérations sur un objet, opérations sur plusieurs objets.
6. Abandonner et réessayer : une bonne idée, avec des limites.

Le livre ne contient ni code ni exercices : tout ce qui est implémenté ici l'est « à la
main », dans le module [`ddia/transactions.py`](../../ddia/transactions.py).

In [1]:
import os
import sqlite3
import tempfile
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter

from ddia import transactions as tr
from ddia.transactions import R, W, run, plot_history
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)

set_style()
plt.rcParams["axes.titlepad"] = 16
# graduations des axes logarithmiques sans mathtext (la police du style n'a pas le signe moins)
LOGFMT = FuncFormatter(lambda v, _: f"{v:g}")
pd.set_option("display.precision", 3)
rng = np.random.default_rng(0)